# Endothelial cell vignette

## Purpose

Explore where endothelial cells come from and whether they have transcriptional memory of their origin.

## Setup

In [ ]:
import devmap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycea as py
import scanpy as sc
import seaborn as sns
import treedata as td
from devmap.config import (
    get_paths,
    meso_lineage_palette,
    set_theme,
)
from devmap.fate import extant_node_attribute
from devmap.plots import plot_program_lineplot, plot_restriction_sankey
from devmap.progenitors import get_fate_progenitors, mark_sibling_descendants
from devmap.utils import load_data, save_plot

set_theme()
base_path, plots_path, results_path = get_paths("vignettes")

## Load data

In [2]:
tdata = load_data("log1p")

Gene programs

In [ ]:
program_results = base_path / "gene_programs" / "results"
program_scores = pd.read_csv(program_results / "program_scores.csv", index_col=0)
tdata.obs = tdata.obs.merge(program_scores, left_index=True, right_index=True)

## Endothelial-restricted clades

In [ ]:
endothelial_types = ["Endothelial cell","Endothelial progenitor","Arterial endothelial cell", "Endocardial cell",
                     "Allantoic endothelial progenitor", "Hemogenic endothelium"]
tdata.obs["is_endothelial"] = tdata.obs["cell_type"].isin(endothelial_types).astype(int)
py.tl.ancestral_states(tdata, "is_endothelial", method="mean")
progenitors = get_fate_progenitors(tdata, "is_endothelial", min_descendants=None)
tdata.obs["is_endothelial"] = tdata.obs["is_endothelial"].astype(bool)

## Identify progenitor origin

In [4]:
cell_type_to_lineage = {
    **dict.fromkeys(["Head paraxial mesoderm", "Pharyngeal mesoderm", "Periocular mesoderm"], "Head mesoderm"),
    **dict.fromkeys(["First heart field", "Second heart field", "Endocardial cell", "Epicardium", "Juxtacardiac field"], "Cardiac mesoderm"),
}

allowed_lineages = {
    "Paraxial mesoderm", "Lateral plate mesoderm", "Cardiac mesoderm",
    "Head mesoderm", "Extraembryonic mesoderm", "Intermediate mesoderm",
}

tdata.obs["meso_lineage"] = (
    tdata.obs["cell_type"].map(cell_type_to_lineage)
    .fillna(tdata.obs["lineage"].astype(str))
    .where(lambda x: x.isin(allowed_lineages), "Other")
)

In [5]:
mark_sibling_descendants(tdata, progenitors)
endo_origin = tdata.obs.query("~is_endothelial").groupby("sibling_descendants")["meso_lineage"].agg(lambda x: x.value_counts().index[0])
tdata.obs["endo_origin"] = tdata.obs.sibling_descendants.map(endo_origin)
progenitors["endo_origin"] = progenitors.node.map(endo_origin)

Umap

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3), dpi = 600)
sc.pl.umap(tdata, color="endo_origin", palette=meso_lineage_palette, ax = ax, frameon=False, title = "")
save_plot(plots_path / "endo_origin_umap.svg", fig, rasterize = True)

Umap zoom

In [ ]:
endo_tdata = tdata[tdata.obs.cell_type.isin(endothelial_types)].copy()
endo_tdata = endo_tdata[endo_tdata.obsm["X_umap"][:, 1] > 16].copy()
fig, ax = plt.subplots(figsize=(2.5, 2.5), dpi = 600)
sc.pl.umap(endo_tdata, color="endo_origin", palette=meso_lineage_palette,
           ax = ax, frameon=False, title = "")
save_plot(plots_path / "endo_origin_umap_zoom.svg", fig, rasterize = True)

## E9.5 origin barplot

In [ ]:
df = tdata.obs.query("is_endothelial & stage == 'E9.5'").copy()

counts = (
    df.groupby("embryo", observed=True)["endo_origin"]
    .value_counts(normalize=True)
    .rename("fraction")
    .reset_index()
)

origin_order = counts["endo_origin"].drop_duplicates().tolist()
if "Other" in origin_order:
    origin_order = [x for x in origin_order if x != "Other"] + ["Other"]

embryo_order = counts["embryo"].drop_duplicates().tolist()

fig, ax = plt.subplots(figsize=(0.35 * len(embryo_order), 1.2), dpi=600)

bottom = np.zeros(len(embryo_order))

for origin in origin_order:
    vals = (
        counts[counts["endo_origin"].eq(origin)]
        .set_index("embryo")
        .reindex(embryo_order)["fraction"]
        .fillna(0)
        .to_numpy()
    )

    ax.bar(
        embryo_order,
        vals,
        bottom=bottom,
        label=origin,
        color=meso_lineage_palette[origin],
        width=0.8,
    )
    bottom += vals

ax.set_ylim(0, 1)
ax.set_ylabel("Endothelial source (%)")
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0%", "25%", "50%", "75%", "100%"])
ax.set_xlabel("")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)

ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()
save_plot(plots_path / "endo_clade_lineage_barplot_by_embryo.svg", fig, rasterize = False)

## Endothelial sibling distribution

In [ ]:
lineages = [
    "Lateral plate mesoderm",
    "Head mesoderm",
    "Paraxial mesoderm",
    "Cardiac mesoderm",
]

stage = "E9.5"
top_n = 3

plot_dfs = []

fig, axes = plt.subplots(
    len(lineages), 2,
    figsize=(2.4, 2.4),
    dpi=600,
    sharex="col",
    sharey="row",
    gridspec_kw={"width_ratios": [1, 1.1], "wspace": 0.08, "hspace": 0.22},
)

for i, lineage in enumerate(lineages):
    foreground = tdata.obs.query(
        "~is_endothelial & stage == @stage & endo_origin.notna() & meso_lineage == @lineage"
    ).copy()

    background = tdata.obs.query(
        "~is_endothelial & stage == @stage & meso_lineage == @lineage"
    ).copy()

    top_subtypes = foreground["cell_subtype"].value_counts().head(top_n).index

    foreground["cell_subtype_plot"] = foreground["cell_subtype"].where(
        foreground["cell_subtype"].isin(top_subtypes), "Other"
    )
    background["cell_subtype_plot"] = background["cell_subtype"].where(
        background["cell_subtype"].isin(top_subtypes), "Other"
    )

    counts = foreground["cell_subtype_plot"].value_counts()
    bg_counts = background["cell_subtype_plot"].value_counts()

    plot_df = (
        pd.DataFrame({
            "count": counts,
            "foreground_frac": counts / counts.sum(),
            "background_frac": bg_counts / bg_counts.sum(),
        })
        .fillna(0)
        .assign(enrichment=lambda x: x["foreground_frac"] / x["background_frac"])
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["enrichment"])
        .sort_values("count", ascending=False)
        .reset_index(names="cell_subtype")
    )
    plot_df["lineage"] = lineage
    plot_dfs.append(plot_df)

    color = meso_lineage_palette[lineage]

    sns.barplot(data=plot_df, y="cell_subtype", x="count", ax=axes[i, 0], color=color, saturation = 1)
    sns.barplot(data=plot_df, y="cell_subtype", x="enrichment", ax=axes[i, 1], color=color, saturation = 1)

    axes[i, 0].set(xlabel="", ylabel="")
    axes[i, 1].set(xlabel="", ylabel="")
    axes[i, 1].axvline(1, color="black", lw=0.8, ls="--")

    for ax in axes[i]:
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(axis="both", labelsize=7)

    axes[i, 1].tick_params(axis="y", left=False, labelleft=False)

axes[-1, 0].set_xlabel("Count")
axes[-1, 1].set_xlabel("Enrichment")

plot_df_all = pd.concat(plot_dfs, ignore_index=True)
plt.tight_layout()
save_plot(plots_path / "endo_sibling_enrichment.svg", fig, rasterize = False)

## Gene expression differences by origin

In [ ]:
lineages = [
    "Extraembryonic mesoderm",
    "Lateral plate mesoderm",
    "Paraxial mesoderm",
    "Head mesoderm",
    "Cardiac mesoderm",
    "Intermediate mesoderm",
]

sc.tl.rank_genes_groups(endo_tdata, groupby="endo_origin", groups = lineages)
markers = sc.get.rank_genes_groups_df(endo_tdata, group = None)
markers["abs_logfoldchanges"] = markers["logfoldchanges"].abs()
top5 = (
    markers
    .query("pvals_adj < 0.05 & abs_logfoldchanges > 1 & group in @lineages")
    .assign(group=lambda x: pd.Categorical(x["group"], categories=reversed(lineages), ordered=True))
    .sort_values(["group", "scores"], ascending=[True, False])
    .groupby("group", observed=True)
    .head(5)
)
top5_gene_dict = {
    lineage: top5.loc[top5["group"].eq(lineage), "names"].tolist()
    for lineage in lineages
}

top5_gene_dict["Shared"] = ["Pecam1", "Cdh5", "Kdr", "Tek", "Flt1"]
tdata_plot = endo_tdata[endo_tdata.obs["endo_origin"].isin(lineages)].copy()
tdata_plot.obs["endo_origin"] = pd.Categorical(
    tdata_plot.obs["endo_origin"],
    categories=lineages,
    ordered=True,
)
sc.pl.heatmap(
    tdata_plot,
    groupby="endo_origin",
    var_names=top5_gene_dict,
    figsize=(3.5, 2.4),
    show=False,
)
save_plot(plots_path / "endo_origin_marker_heatmap.svg", plt.gcf(), rasterize = True)

## Specification timing

In [ ]:
lineages = [
    "Extraembryonic mesoderm",
    "Cardiac mesoderm",
    "Head mesoderm",
    "Lateral plate mesoderm",
    "Paraxial mesoderm",
    "Intermediate mesoderm",
]
fig, ax = plt.subplots(figsize=(1,2), dpi = 600)
sns.violinplot(data = progenitors.query("stage == 'E9.5' & fate_descendants > 1"), y = "endo_origin", x = "time", hue = "endo_origin",
               order = lineages, palette = meso_lineage_palette, inner = "quart", linewidth=0.8, saturation = 1, ax = ax, legend = False)
plt.xlabel("Inferred restriction time");
plt.ylabel("Fate");
save_plot(plots_path / "endo_restriction_violin.svg", fig)

## Specification dynamics

In [ ]:
e95_tdata = tdata[tdata.obs.stage == "E9.5", []].copy()
e95_tdata.obs.loc[e95_tdata.obs["is_endothelial"], "meso_lineage"] = pd.NA
e95_tdata.obs.loc[(e95_tdata.obs["germ_layer"] == "Mesoderm") & (e95_tdata.obs["meso_lineage"] == "Other"), "meso_lineage"] = pd.NA
lineage_assignment = e95_tdata.obs["meso_lineage"].to_dict()
endo_assignment = e95_tdata.obs["is_endothelial"].map({True: "Endothelial", False: "Other"}).to_dict()
for tree in e95_tdata.obst.values():
    devmap.fate.annotate_commitment(tree, [lineage_assignment, endo_assignment], threshold=0.9, exclude_categories=["Other"])
paths = devmap.fate.extant_transition_count(e95_tdata[e95_tdata.obs.is_endothelial], 
    key = "commitment", depth_key="time", bins = np.arange(0, 10, .5))

In [ ]:
fig, ax = plt.subplots(figsize=(3.2, 2),dpi=600)
sankey_order = ["Uncommitted"] + list(meso_lineage_palette.keys()) + ["Endothelial"]
sankey_palette = {**meso_lineage_palette, **{"Uncommitted": "lightgray", "Endothelial": "#00ddff"}}
plot_restriction_sankey(paths.query("source_time > 3"), normalize = True, order = sankey_order, palette = sankey_palette, ax = ax)
plt.xticks([1,5,9,13], ["4", "6", "8", "10"]);
plt.xlim(0, 13)
plt.xticks(rotation=0)
save_plot(plots_path / "endothelial_sankey.svg", fig)

## Endothelial program over time

In [ ]:
py.tl.ancestral_states(tdata, "P34", method="mean")
sampled_tdata = tdata[tdata.obs.query("is_endothelial")
    .groupby("endo_origin", group_keys=False, observed = True)
    .apply(lambda x: x.sample(min(len(x), 100))).index].copy()
origin_scores = extant_node_attribute(sampled_tdata,key = "P34", depth_key="time",paths = True,
                           bins = np.arange(0, 10, .5), return_type = "dataframe")
origin_scores["endo_origin"] = origin_scores["leaf"].map(tdata.obs["endo_origin"])
df = origin_scores.query("endo_origin != 'Other'").groupby(["endo_origin", "time"])["P34"].mean().reset_index()
fig, ax = plt.subplots(figsize=(1.8, 1.8), dpi = 600)
plot_program_lineplot(df, df, y="P34", ax=ax,  palette = meso_lineage_palette, hue = "endo_origin")
# remove legend
ax.get_legend().remove()
save_plot(plots_path / "endo_origin_P34_lineplot.svg", fig, rasterize = True)

## Somite program expression

In [ ]:
fig, ax = plt.subplots(figsize=(2.5, 2.5), dpi = 600)
sc.pl.umap(tdata, color="P45", cmap = "PuOr_r", vmin = -1, vmax = 1, ax = ax, frameon=False, title = "")
save_plot(plots_path / "P45_umap.svg", fig, rasterize = True)

Zoom in on endothelial cells

In [ ]:
fig, ax = plt.subplots(figsize=(2.5, 2.5), dpi = 600)
sc.pl.umap(endo_tdata, color="P45", cmap = "PuOr_r", vmin = -1, vmax = 1, frameon=False, ax = ax, title = "")
save_plot(plots_path / "P45_endo_umap.svg", fig, rasterize = True)

## Example clades

In [ ]:
fig, ax = plt.subplots(figsize=(1.2, 3.6), dpi = 600)
clone = 'E9.5-R2-C1'
endo_color = "#00ddff"
nodes =progenitors.query("clone == @clone").node.unique().tolist()
py.pl.tree(tdata, depth_key="time", tree = clone, ax = ax, branch_linewidth=.5)
py.pl.nodes(tdata,tree = clone, nodes = nodes, ax = ax, color = endo_color, size = 10, outline_width=.3)
save_plot(plots_path / "endo_progenitor_tree.svg", fig, rasterize=True)

In [ ]:
for node in ["XRAchitj","Ft0zFL33","aq6SOFXm"]:
    subset = tdata[tdata.obs.grandparent_descendants == node,[]].copy()
    fig, ax = plt.subplots(figsize=(1.5,subset.shape[0]/20), dpi=600)
    subset.obs.loc[subset.obs.is_endothelial == 'True', "meso_lineage"] = subset.obs.query("clade_lineage.notnull()").clade_lineage.mode()[0]
    py.pl.tree(subset, depth_key="time", ax = ax, branch_linewidth=0.5)
    py.pl.annotation(subset, keys = "is_endothelial", palette = ["lightgray", endo_color], label = False, ax = ax, gap = .05, legend = True, width = .15)
    py.pl.annotation(subset, keys = "meso_lineage", palette = meso_lineage_palette, label = False, ax = ax, gap = .05, legend = True, width = .15)
    save_plot(plots_path / f"endo_tree_{node}.svg", fig, rasterize=False)